# 05 风险验证：通过率下降是否对应更好风险？

## 本步目标

1. 把审批通过率和 DPD30 放在一起看。
2. 判断“低通过率渠道”是否真的低风险。
3. 理解选择偏差：我们只能看到获批客户的风险，看不到被拒绝客户的风险。

## 口径提醒

- DPD30：单期最大逾期天数 ≥ 30。
- 这里用的是“获批客户”的还款表现，存在选择偏差。
- 不能用它直接证明“收紧审批一定降低了风险”。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from IPython.display import display
from src.db import query_df
from src.charts import monthly_trend, channel_bar, contribution_bar, risk_scatter

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 50)
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

## 1. 渠道通过率与 DPD30

In [2]:
channel = query_df("SELECT `渠道类型`,`进件量`,`审批通过率` FROM `v_渠道月度` WHERE `月份`='2026-09'")
risk = query_df("SELECT * FROM `v_还款风险`")
merged = channel.merge(risk, on="渠道类型", how="left")
display(merged.sort_values("进件量", ascending=False))
risk_scatter(merged.dropna(subset=["DPD30率"]), FIG / "04_通过率与风险.png")

,渠道类型,进件量,审批通过率,有还款记录合同数,DPD30合同数,DPD30率
4,Credit and cash offices,23430,0.0784,251520,3776,0.0150
2,Contact center,4939,0.0433,20144,966,0.0480
3,Country-wide,4384,0.8764,382101,11228,0.0294
6,Stone,1397,0.8812,184034,5440,0.0296
5,Regional / Local,1075,0.8744,91583,1736,0.0190
0,AP+ (Cash loan),468,0.2308,27292,503,0.0184
1,Channel of corporate sales,319,0.2288,1991,12,0.0060


WindowsPath('E:/ai数据分析工作/consumer-finance-approval-analysis/outputs/figures/04_通过率与风险.png')

## 2. 通过率最低的渠道，DPD30 是否也最低？

In [3]:
display(merged.sort_values("审批通过率").head(6))

,渠道类型,进件量,审批通过率,有还款记录合同数,DPD30合同数,DPD30率
2,Contact center,4939,0.0433,20144,966,0.0480
4,Credit and cash offices,23430,0.0784,251520,3776,0.0150
1,Channel of corporate sales,319,0.2288,1991,12,0.0060
0,AP+ (Cash loan),468,0.2308,27292,503,0.0184
5,Regional / Local,1075,0.8744,91583,1736,0.0190
3,Country-wide,4384,0.8764,382101,11228,0.0294


## 3. 账龄迁徙：同 cohort 比较

In [4]:
try:
    vintage = query_df("SELECT * FROM `v_账龄迁徙` WHERE `账龄月` IN (0,3,6,12) ORDER BY `放款cohort`,`账龄月`")
    display(vintage.tail(20))
except Exception as exc:
    print("账龄迁徙表暂未生成，先看渠道风险。原因：", exc)

,放款cohort,账龄月,贷款数,累计DPD30率
347,2025-12,3,18015,0.0025
348,2025-12,6,13313,0.0048
349,2026-01,0,20196,0.0003
350,2026-01,3,18075,0.0023
351,2026-01,6,13712,0.0047
352,2026-02,0,17729,0.0006
353,2026-02,3,15864,0.0024
354,2026-02,6,12031,0.0049
355,2026-03,0,18171,0.0007
356,2026-03,3,16202,0.0022


## 本步结论

审批通过率低的渠道并不必然对应更低 DPD30；Contact center 通过率 4.33%、DPD30 4.80%，说明低通过不等于低风险。风险验证只能覆盖获批客户，存在选择偏差。